# DBSCAN tuning (Phase 4)

Diagnostic notebook for choosing `eps` and `min_samples`. The committed
values live in `config/detection.yaml`; production code never hard-codes
them. Run this after `make features` (sample mode is fine).


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from kneed import KneeLocator
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sqlalchemy import text

from chainlens.db import get_pg_engine
from chainlens.detection import load_detection_config
from chainlens.features import get_scaled_feature_columns

engine = get_pg_engine()
columns, scaler_version = get_scaled_feature_columns(engine)
print(columns)
print(scaler_version)

In [ ]:
query = (
    "SELECT address, "
    + ", ".join(columns)
    + " FROM address_features WHERE tx_count_total >= 3 ORDER BY address"
)
with engine.connect() as conn:
    frame = pd.read_sql(text(query), conn)

X = frame[columns].to_numpy(dtype=float)
print("eligible addresses:", len(frame))
print("matrix:", X.shape, "finite:", np.isfinite(X).all())
assert np.isfinite(X).all()

In [ ]:
config = load_detection_config()["dbscan"]
suggested_min_samples = max(3, math.ceil(0.02 * len(frame)))
min_samples = int(config["min_samples"])
eps = float(config["eps"])
print("suggested min_samples:", suggested_min_samples)
print("configured min_samples:", min_samples)
print("configured eps:", eps)
assert min_samples == suggested_min_samples, "re-tune after changing eligibility"

In [ ]:
# k-distance curve: the prompt's starting recipe uses n_neighbors=min_samples.
distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
k_distances = np.sort(distances[:, -1])

knee = KneeLocator(
    range(len(k_distances)),
    k_distances,
    curve="convex",
    direction="increasing",
    S=1.0,
)
print("knee index:", knee.knee)
print("knee eps:", knee.knee_y)
print("configured eps:", eps)

plt.figure(figsize=(8, 4))
plt.plot(range(len(k_distances)), k_distances, marker=".", linewidth=1)
plt.axhline(eps, color="tab:red", linestyle="--", label=f"config eps={eps}")
plt.xlabel("points sorted by distance")
plt.ylabel(f"distance to {min_samples}th neighbour")
plt.legend()
plt.title(f"k-distance (scaler {scaler_version})")
plt.show()

In [ ]:
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist(), strict=False)))
print("clusters:", len(set(labels.tolist()) - {-1}))
print("noise:", int((labels == -1).sum()))

In [ ]:
projection = PCA(n_components=2, random_state=42).fit_transform(X)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    projection[:, 0],
    projection[:, 1],
    c=labels,
    cmap="tab10",
    s=28,
    alpha=0.85,
)
plt.colorbar(scatter, label="cluster id (-1 = noise)")
plt.title(f"PCA projection of eligible addresses (eps={eps})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

Recorded choice for the fixture build: `eps=3.195323`,
`min_samples=5`, `n_eligible=216`. The knee is diagnostic, not an
automatic production write: review the curve and commit the chosen values
to `config/detection.yaml`.
